In [ ]:
import os
import pandas as pd
import anthropic
from dotenv import load_dotenv

# 1. SETUP & PATHS
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
BASE_PATH = "/Users/rajhomedesktop/anaconda_projects/Anthropic AI"
SKILLS_PATH = os.path.join(BASE_PATH, "Skills")
EXCEL_PATH = '/Users/rajhomedesktop/Desktop/TimberLens-Creations LLC/TimberLens-Creations_Business_Plan_FY2026.xlsx'

# 2. LOAD BUSINESS CONTEXT
xl = pd.ExcelFile(EXCEL_PATH)
full_plan_context = ""
for sheet in xl.sheet_names:
    df_sheet = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
    full_plan_context += f"\n--- {sheet} ---\n{df_sheet.to_string()}\n"

# 3. RECOVERY HELPER
def get_skill(filename, fallback):
    path = os.path.join(SKILLS_PATH, filename)
    if os.path.exists(path):
        with open(path, "r") as f: return f.read()
    return fallback

# 4. THE REGISTRY
skills_registry = {
    "Manager.md": get_skill("Manager.md", "Manager template..."),
    "Master Artisan.md": get_skill("Master Artisan.md", "Artisan template..."),
    "SupplyChain.md": get_skill("SupplyChain.md", "GPS & Logistics template..."), # This will keep GPS updates
    "Finance.md": get_skill("Finance.md", "Finance template..."),
    "WebDev.md": get_skill("WebDev.md", "WebDev template..."),
    "Inventory.md": get_skill("Inventory.md", "Inventory template..."),
    "Marketing.md": get_skill("Marketing.md", "Marketing template..."),
    "System_State.md": f"# TIMBERLENS STATE\nSync: {pd.Timestamp.now()}\n{full_plan_context}"
}

# 5. EXECUTE SYNC
print(f" Syncing TimberLens Intelligence...")
for filename, content in skills_registry.items():
    # Use 'filename' here, because that's what is in your loop header!
    file_path = os.path.join(SKILLS_PATH, filename) 
    
    with open(file_path, "w") as f:
        f.write(content)
    print(f"✅ Skill Synced: {filename}")

In [ ]:
#VERIFICATION TEST CODE

import os

# 1. Define the files we expect to see
expected_files = [
    "Manager.md", "Finance.md", "WebDev.md", "SupplyChain.md", 
    "Inventory.md", "Marketing.md", "Master Artisan.md", "System_State.md"
]

print(f"🔍 Auditing Skills Directory: {SKILLS_PATH}\n" + "="*50)

missing = []
for file in expected_files:
    file_path = os.path.join(SKILLS_PATH, file)
    
    if os.path.exists(file_path):
        # Check if the file is empty or has content
        size = os.path.getsize(file_path)
        status = "✅ PASS" if size > 100 else "⚠️ WARNING (Short file)"
        
        # Peek at the first line to confirm it's the right content
        with open(file_path, 'r') as f:
            first_line = f.readline().strip()[:40]
            
        print(f"{file.ljust(18)} | {status} | {size:>6} bytes | Peek: {first_line}...")
    else:
        print(f"{file.ljust(18)} | ❌ MISSING")
        missing.append(file)

print("="*50)
if not missing:
    print("✨ ALL CLEAR: Your AI Team is synchronized and ready for the App.")
else:
    print(f"🚨 ALERT: {len(missing)} files are missing. Re-run the Sync block.")

In [ ]:
import os

# --- CONFIGURATION ---
MODEL_NAME = "claude-opus-4-6" # Set once here

def ask_skill(skill_name, user_input, extra_context=""):
    skill_path = os.path.join(SKILLS_PATH, f"{skill_name}.md")
    state_path = os.path.join(SKILLS_PATH, "System_State.md")
    
    with open(skill_path, "r") as f:
        skill_prompt = f.read()
    with open(state_path, "r") as f:
        system_state = f.read()
        
    combined_system = f"{skill_prompt}\n\n### LIVE BUSINESS DATA ###\n{system_state}\n{extra_context}"
    
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1500,
        system=combined_system,
        messages=[{"role": "user", "content": user_input}]
    )
    return response.content[0].text

def timberlens_orchestrator(user_prompt):
    print(f"🧠 Manager: Reviewing request...")
    
    # 1. Manager defines the 'Meeting Agenda'
    agenda_query = f"The user asked: '{user_prompt}'. Who do we need to consult and what specific data should we extract from our 2026 plan?"
    agenda = ask_skill("Manager", agenda_query)
    
    # 2. Sequential Consultation (The Discussion)
    context_accumulator = f"MANAGER'S AGENDA: {agenda}\n"
    
    # Define which agents to 'call' based on the request
    for skill in ["Master Artisan", "SupplyChain", "Finance"]:
        print(f"💬 Consulting {skill}...")
        contribution = ask_skill(skill, user_prompt, extra_context=context_accumulator)
        context_accumulator += f"\n{skill.upper()} INPUT: {contribution}\n"

    # 3. Final Manager Synthesis
    print(f"⚖️ Manager: Synthesizing final response...")
    final_output = ask_skill("Manager", 
                            f"Summarize the team's findings and answer the user's original request: {user_prompt}", 
                            extra_context=context_accumulator)
    return final_output

# --- TEST ---
user_request = "What are the wood variants used and why are they best in market in price, quality, and CAS design? Provide a 2026 projection table."
print(timberlens_orchestrator(user_request))